# gdmbayes: basic usage

Fit the frequentist `GDM` and the Bayesian `spGDMM` to a small synthetic dataset,
inspect sampling diagnostics, and predict dissimilarities for new sites. The data
mimic a marine survey: 30 sites with coordinates, three environmental predictors
(temperature, salinity, depth) and Bray–Curtis dissimilarities computed from random
biomass tables, so the fitted effects mean nothing — only the workflow matters here.

Runs in about a minute; the Bayesian fit uses a deliberately tiny sampler budget.

In [1]:
import warnings


import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist

from gdmbayes import GDM, GDMPreprocessor, ModelConfig, SamplerConfig, spGDMM

# Third-party numerical warnings that are expected for the tiny sampler budget used here.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="arviz_stats")

## 1. Generate synthetic data

In [2]:
np.random.seed(42)
n_sites = 30

X = pd.DataFrame({
    "xc": np.random.uniform(0, 100, n_sites),
    "yc": np.random.uniform(0, 100, n_sites),
    "time_idx": np.zeros(n_sites, dtype=int),
    "temp": np.random.uniform(5, 20, n_sites),
    "salinity": np.random.uniform(30, 35, n_sites),
    "depth": np.random.uniform(0, 200, n_sites),
})

biomass = np.random.exponential(1, (n_sites, 20))
y = pdist(biomass, "braycurtis")
y = np.clip(y, 1e-8, 1 - 1e-8)

print(f"Sites: {n_sites}  |  Pairs: {len(y)}  |  y range: [{y.min():.3f}, {y.max():.3f}]")

Sites: 30  |  Pairs: 435  |  y range: [0.286, 0.682]


## 2. Frequentist GDM

In [3]:
gdm_model = GDM(geo=True)
gdm_model.fit(X, y)

print(f"Deviance explained: {gdm_model.explained_:.4f}")
print(f"Predictor importance: {gdm_model.predictor_importance_}")
print(f"Coefficients shape: {gdm_model.coef_.shape}")

# Predict pairwise dissimilarities
preds = gdm_model.predict(X)
print(f"Prediction range: [{preds.min():.3f}, {preds.max():.3f}]")

Deviance explained: 0.0573
Predictor importance: {'temp': 0.012415880014470729, 'salinity': 0.1617513508344624, 'depth': 0.01221178467991639, 'geo': 0.12157991830111642}
Coefficients shape: (29,)
Prediction range: [0.451, 0.569]


## 3. Bayesian spGDMM (minimal sampling for example speed)

In [4]:
sampler_cfg = SamplerConfig(draws=100, tune=100, chains=2, nuts_sampler="pymc", progressbar=True)

model = spGDMM(
    preprocessor=GDMPreprocessor(deg=3, knots=2),
    model_config=ModelConfig(variance="homogeneous", spatial_effect="none"),
    sampler_config=sampler_cfg,
)

model.fit(X, y, random_seed=42)
idata = model.idata_

print(f"Posterior samples: {idata.posterior.dims}")
print(f"Variables: {list(idata.posterior.data_vars.keys())}")

# Sampling diagnostics
print("\nSampling Diagnostics")
diag = model.summarise_sampling()
print(diag.head(10))

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [beta_0, beta, alpha, beta_dist, sigma2]


Output()

Sampling 2 chains for 100 tune and 100 draw iterations (200 + 200 draws total) took 0 seconds.


...

Sampling: [alpha, beta, beta_0, beta_dist, log_y, sigma2]


Sampling: [log_y]


Output()

Posterior samples: Frozen(ChainMap({'chain': 2, 'draw': 100, 'feature': 3, 'basis_function': 5, 'beta_dist_dim_0': 5}, {}))
Variables: ['beta_0', 'beta', 'alpha', 'beta_dist', 'sigma2']

Sampling Diagnostics
                    mean          sd ess_bulk ess_tail r_hat
beta_0               0.4   0.0868992        2        5  2.34
beta[temp, 1]        0.2           0        2        5  2.10
beta[temp, 2]        0.2    0.115626        2        5  2.26
beta[temp, 3]        0.2           0        2        6  2.08
beta[temp, 4]      0.153  0.00269555        2        5  2.21
beta[temp, 5]        0.2           0        2        5  1.97
beta[salinity, 1]   0.12       0.021        2       15  1.92
beta[salinity, 2]    0.2   0.0841386        2        6  1.95
beta[salinity, 3]   0.29           0        3       13  1.89
beta[salinity, 4]    0.3   0.0786574        3       19  1.92


/tmp/ipykernel_1149729/3326807426.py:17: ConvergenceWarning: 25 parameter(s) with R-hat > 1.01
  diag = model.summarise_sampling()
/tmp/ipykernel_1149729/3326807426.py:17: ConvergenceWarning: 25 parameter(s) with ESS_bulk < 100
  diag = model.summarise_sampling()


## 4. Predict on new data

In [5]:
n_pred = 10
X_pred = pd.DataFrame({
    "xc": np.random.uniform(0, 100, n_pred),
    "yc": np.random.uniform(0, 100, n_pred),
    "time_idx": np.zeros(n_pred, dtype=int),
    "temp": np.random.uniform(5, 20, n_pred),
    "salinity": np.random.uniform(30, 35, n_pred),
    "depth": np.random.uniform(0, 200, n_pred),
})

post_pred = model.predict_posterior(X_pred, extend_idata=False, combined=True)
print(f"Posterior predictive shape: {post_pred.shape}")

# model.save("spgdmm_model.nc")

/tmp/ipykernel_1149729/413139247.py:11: UserWarning: 1 env values were clipped to predictor_mesh bounds.
  post_pred = model.predict_posterior(X_pred, extend_idata=False, combined=True)


Sampling: [log_y]


Output()

Posterior predictive shape: (45, 200)
